# 1D J2J2 inference: Euclidean GRU (seed 111-555)

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../../1_hypnqs_torch_poincare/utility_poincare')
from j1j2_hyprnn_train_loop import *
numsamples = 10000

GPU Available:  False


In [4]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  

def define_load_test(wf, numsamples,path_to_weights, Ee, clipped_e=False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [5]:
N=100
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname=f'results_EuclideanGRU'

## J2=0.0

In [6]:
J2_ = 0.0
J2 = +J2_ * np.ones(syssize)
Ee_00=-44.12773986967251

In [8]:
#No convergence
seed=111
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-10.333499908447266-0.024900000542402267j), var E = 19.578399658203125
DMRG energy  is -44.1277
Time taken=0.025 hrs


In [9]:
seed=222
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.37080001831055-0.007400000002235174j), var E = 1.0508999824523926
DMRG energy  is -44.1277
Time taken=0.058 hrs


In [10]:
seed=333
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.85029983520508-0.005799999926239252j), var E = 0.602400004863739
DMRG energy  is -44.1277
Time taken=0.085 hrs


In [11]:
seed=444
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.65449905395508-0.00039999998989515007j), var E = 0.5227000117301941
DMRG energy  is -44.1277
Time taken=0.084 hrs


In [8]:
seed=555
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.88869857788086-0.0003000000142492354j), var E = 2.321500062942505
DMRG energy  is -44.1277
Time taken=0.146 hrs


## J2=0.2

In [9]:
J2_ = 0.2
J2 = +J2_ * np.ones(syssize)
Ee_02=-40.7388

In [10]:
# Earlier training in April
seed=111
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}_o/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.95159912109375+0.006099999882280827j), var E = 0.7871000170707703
DMRG energy  is -40.7388
Time taken=0.043 hrs


In [6]:
seed=222
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.90140151977539-0.008299999870359898j), var E = 1.273900032043457
DMRG energy  is -40.7388
Time taken=0.185 hrs


In [11]:
seed=333
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.96160125732422-0.002199999988079071j), var E = 0.24539999663829803
DMRG energy  is -40.7388
Time taken=0.042 hrs


In [12]:
seed=444
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.93989944458008+0.000699999975040555j), var E = 0.7687000036239624
DMRG energy  is -40.7388
Time taken=0.042 hrs


In [13]:
seed=555
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.81549835205078-0.0012000000569969416j), var E = 0.5591999888420105
DMRG energy  is -40.7388
Time taken=0.042 hrs


## J2=0.5 

In [14]:
J2_ = 0.5
J2 = +J2_ * np.ones(syssize)
Ee_05=37.5000
fname = f'results_EuclideanGRU/J2={J2_}/local_machine'

In [16]:
#Best model saved at epoch 616 with best E=-36.97652-0.16221j, varE=15.68240
seed=111
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.4734992980957+0.0026000000070780516j), var E = 1.086899995803833
DMRG energy  is 37.5
Time taken=0.122 hrs


In [17]:
#Best model saved at epoch 827 with best E=-37.51801+0.00137j, varE=0.01488
seed=222
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.46799850463867-0.0005000000237487257j), var E = 0.05570000037550926
DMRG energy  is 37.5
Time taken=0.096 hrs


In [22]:
seed=333 
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-30.556299209594727+0.006099999882280827j), var E = 3.136899948120117
DMRG energy  is 37.5
Time taken=0.068 hrs


In [23]:
#Best model saved at epoch 757 with best E=-37.53916-0.00253j, varE=0.09154
seed=444 
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.459800720214844-9.999999747378752e-05j), var E = 0.10440000146627426
DMRG energy  is 37.5
Time taken=0.072 hrs


In [30]:
seed=555 
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.018001556396484-9.999999747378752e-05j), var E = 0.05860000103712082
DMRG energy  is 37.5
Time taken=0.056 hrs


## J2=0.8 

In [17]:
J2_ = 0.8
J2 = +J2_ * np.ones(syssize)
Ee_08=-42.07006297371643
fname = f'results_EuclideanGRU/J2={J2_}/local_machine'

In [18]:
# Earlier training in April
seed=111
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}_o/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.87860107421875-0.0005000000237487257j), var E = 1.8287999629974365
DMRG energy  is -42.0701
Time taken=0.046 hrs


In [20]:
seed=222
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.40909957885742+0.0044999998062849045j), var E = 0.8184999823570251
DMRG energy  is -42.0701
Time taken=0.049 hrs


In [21]:
seed=333
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.27870178222656+0.006200000178068876j), var E = 0.7833999991416931
DMRG energy  is -42.0701
Time taken=0.078 hrs


In [22]:
seed=444
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.524600982666016+0.007199999876320362j), var E = 1.073699951171875
DMRG energy  is -42.0701
Time taken=0.073 hrs


In [23]:
seed=555
eGRU_00 = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=seed)
e_wl= f'{fname}/seed={seed}/N100_J1=1.0|J2={J2_}_EuclGRU_70_rmax=None_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(eGRU_00, numsamples, e_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.89540100097656+0.0017999999690800905j), var E = 1.732800006866455
DMRG energy  is -42.0701
Time taken=0.077 hrs
